# Trust-Aware AsymMirai — Uncertainty Quantification (v2)

Runs the full pipeline on Colab: **download → DICOM→16-bit PNG → exam table with labels → (optional) fine-tune heads → MC Dropout / paired TTA / ensembles → evaluation**.

Before running:
1. Put this repo at `MyDrive/dissertation/AsymMirai_UQ` (or change `REPO` below).
2. Put the trained AsymMirai weights at `MyDrive/dissertation/weights/asymmirai_full_model.pt` (download link in `AsymMirai/README.md`).
3. Put the EMBED tables in `MyDrive/dissertation/data/tables/`.
4. Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` as **Colab Secrets** (key icon, left sidebar). Never paste keys into a cell.
5. Runtime → Change runtime type → **GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DATA_ROOT = '/content/drive/MyDrive/dissertation'          # holds data/, weights/, outputs/
REPO = f'{DATA_ROOT}/AsymMirai_UQ'
os.environ['ASYMMIRAI_UQ_ROOT'] = DATA_ROOT
os.chdir(REPO); sys.path.insert(0, REPO)
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm lifelines boto3 pyyaml

In [ ]:
from uq.config import load_config, set_seed
cfg = load_config('configs/uq_config.yaml')
set_seed(cfg['seed'])
print('root:', cfg.root_path)
print('weights found:', cfg.path('weights', 'asymmirai_full_model').exists())

## 1. Download DICOMs (skip if already on Drive)
Credentials are read from Colab Secrets. Set `max_files` to download a small subset first.

In [ ]:
from uq.data import download_embed
# n = download_embed.download(cfg, max_files=400)   # uncomment to run
# print(n, 'files')

## 2. DICOM → 16-bit PNG
Min-max window to the full 16-bit range (Mirai's `dcmj2pnm +on2 --min-max-window` convention), MONOCHROME1 inverted. Resumable.

In [ ]:
from uq.data import dicom_to_png
index = dicom_to_png.convert_all(cfg, workers=4)
index.head()

## 3. Exam table + labels
Grouped by accession number (true exam), standard 2D CC/MLO views only, `years_to_cancer` from the clinical table, patient-level calib/test split.

In [ ]:
from uq.data import build_metadata
exams = build_metadata.build(cfg)
exams.drop_duplicates('exam_id')[['split']].value_counts()

## 4. (Optional) Fine-tune heads with dropout / build an ensemble
MC Dropout is only principled if the model was trained with dropout. This re-fits the small AsymMirai head (backbone frozen) with paired dropout. Needs `split.finetune_fraction > 0` in the config (then re-run step 3). Several seeds → deep-ensemble members.

In [ ]:
# from uq import finetune_head
# from uq.config import get_device
# out = cfg.path('outputs') / 'finetuned_heads'; out.mkdir(parents=True, exist_ok=True)
# heads = [finetune_head.train_one(cfg, s, epochs=5, lr=1e-3, horizon=5, device=get_device(),
#                                  out_dir=out, checkpoint=None, bootstrap=True) for s in range(5)]
# cfg['paths']['weights']['ensemble_members'] = [str(h) for h in heads]
# cfg['uq']['methods'].append('ensemble')

## 5. Uncertainty quantification
Run on a small `limit` first and check the sanity line: clean predictions must vary across exams.

In [ ]:
from uq import inference
run_dir = inference.run(cfg, split='all', limit=20, run_name='sanity_check')
import json; print(json.load(open(run_dir / 'run_info.json'))['conditions']['clean'])

In [ ]:
run_dir = inference.run(cfg, split='all', run_name='full_run')

## 6. Evaluation
Thresholds and conformal quantiles fitted on `calib`, everything reported on `test`.

In [ ]:
from uq import evaluate
from IPython.display import Markdown, Image, display
ev = evaluate.run(cfg, run_dir)
display(Markdown((ev / 'report.md').read_text()))
for f in ['roc.png', 'reliability.png', 'risk_coverage_auc.png', 'risk_coverage_brier.png', 'uncertainty_hist.png',
          'confidence_vs_uncertainty.png', 'localisation_uncertainty.png', 'subgroup_referral.png',
          'conformal_sets.png', 'robustness_uncertainty.png', 'robustness_auc.png']:
    if (ev / f).exists():
        display(Image(str(ev / f), width=650))

## 7. ConvNeXt backbone (extension)
Set `model.backbone: convnext_tiny` in the config, run step 4 to fit its head (asymmetry statistics are initialised from data), then run step 5 with `checkpoint=<head .pt>` and compare the reports.

## 8. Demo app
`streamlit run app/streamlit_app.py -- --config configs/uq_config.yaml` (locally, or on Colab with a tunnel).